# NBA ATS ROI

Purpose: reproduce the portfolio `/api/sports-edges/ats-weekly` calculation for NBA and turn it into notebook-backed evidence.

This notebook joins scored Supabase `games` rows to the latest `model_predictions`, computes ATS wins/losses/pushes using the same home-margin convention as the portfolio API, and summarizes flat-unit ROI at -110.

Headline ATS uses `my_spread` vs final margin for every graded game. **Edge bucket tables** use only rows with a numeric `book_spread` (otherwise `edge_vs_book` is undefined). The summary dict includes `graded_missing_book_spread` and `ats_where_book_line_present` for the filtered subset.

Source contracts:

- Supabase serving inventory: `plans/docs/supabase_tables/personal_portfolio_project_tables.json`
- Portfolio implementation: `personal-portfolio/app/api/sports-edges/ats-weekly/route.ts`
- Planned metric exports: `nba_ats_record_2025_26`, `nba_flat_roi_2025_26`, `nba_roi_by_edge_bucket`



In [1]:
LEAGUE = "NBA"
SEASON = None  # inferred from current date if None
EDGE_BUCKETS = [-99, -8, -5, -3, -1, 1, 3, 5, 8, 99]
WIN_PROFIT_AT_MINUS_110 = 100 / 110


In [2]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    DATA_CORE = ROOT.parent
else:
    DATA_CORE = ROOT
if str(DATA_CORE) not in sys.path:
    sys.path.insert(0, str(DATA_CORE))

from src.utils.supabase_pg import create_pg_connection, load_supabase_credentials

def infer_current_season(league: str) -> int:
    now = datetime.now(timezone.utc)
    return now.year - 1 if now.month < 8 else now.year

season = SEASON or infer_current_season(LEAGUE)
season


2025

In [3]:
QUERY = """
WITH latest_predictions AS (
  SELECT
    p.game_id,
    p.my_spread,
    p.my_home_win_prob,
    p.model_version,
    p.asof_ts,
    ROW_NUMBER() OVER (PARTITION BY p.game_id ORDER BY p.asof_ts DESC) AS rn
  FROM model_predictions p
)
SELECT
  g.id AS game_id,
  g.league,
  g.season,
  g.week,
  g.game_time_utc,
  g.home_team,
  g.away_team,
  g.book_spread,
  g.home_score,
  g.away_score,
  p.my_spread,
  p.my_home_win_prob,
  p.model_version,
  p.asof_ts
FROM games g
JOIN latest_predictions p ON p.game_id = g.id AND p.rn = 1
WHERE g.league = %s
  AND g.season = %s
  AND g.home_score IS NOT NULL
  AND g.away_score IS NOT NULL
ORDER BY g.game_time_utc
"""

def load_scored_predictions(league: str, season: int) -> pd.DataFrame:
    creds = load_supabase_credentials()
    missing = [key for key in ("url", "db_password") if not creds.get(key)]
    if missing:
        print(f"Missing Supabase credentials: {missing}. Returning empty frame.")
        return pd.DataFrame()
    conn = create_pg_connection(
        supabase_url=creds["url"],
        password=creds["db_password"],
        host_override=creds.get("db_host"),
        port=creds["db_port"],
        database=creds["db_name"],
        user=creds["db_user"],
    )
    try:
        return pd.read_sql_query(QUERY, conn, params=(league, season))
    finally:
        conn.close()

raw = load_scored_predictions(LEAGUE, season)
raw.head()


/tmp/ipykernel_65167/4220712856.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(QUERY, conn, params=(league, season))


,game_id,league,season,week,game_time_utc,home_team,away_team,book_spread,home_score,away_score,my_spread,my_home_win_prob,model_version,asof_ts
0,f7bf52c6-f94f-4b15-8e90-74574dff19ce,NBA,2025,None,2026-04-20 00:00:00+00:00,DEN,MIN,1.0,114,119,-9.885484,0.751648,v3,2026-04-20 15:18:02.057812+00:00
1,6c87d641-c1c1-45c4-a2fd-9202351cc966,NBA,2025,None,2026-04-20 00:00:00+00:00,NY,ATL,NaN,106,107,-3.225038,0.592149,v3,2026-04-20 15:18:02.057812+00:00
2,4d127ca3-b4dd-4b2d-8455-bbe4eb58cc0c,NBA,2025,None,2026-04-20 00:00:00+00:00,CLE,TOR,3.5,115,105,-6.797049,0.682773,v3,2026-04-20 15:18:02.057812+00:00
3,261b81e1-5271-4ee7-a7bb-efb4ed4c0749,NBA,2025,None,2026-04-21 00:00:00+00:00,LAL,HOU,-9.5,101,94,1.590512,0.462054,v3,2026-04-21 15:14:46.994036+00:00
4,bdfaefcb-daf5-4269-af9c-1ad109b9236c,NBA,2025,None,2026-04-21 00:00:00+00:00,BOS,PHI,7.5,97,111,-6.948005,0.686661,v3,2026-04-21 15:14:46.994036+00:00


In [4]:
def add_ats_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    numeric_cols = ["home_score", "away_score", "my_spread", "book_spread", "my_home_win_prob"]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["actual_margin"] = out["home_score"] - out["away_score"]
    out["cover_margin"] = out["actual_margin"] + out["my_spread"]
    out["ats_result"] = np.select(
        [out["cover_margin"].abs() < 1e-3, out["cover_margin"] > 0],
        ["push", "win"],
        default="loss",
    )
    out["edge_vs_book"] = out["my_spread"] - out["book_spread"]
    out["home_win_actual"] = (out["actual_margin"] > 0).astype(int)
    return out

games = add_ats_columns(raw)
games[["game_time_utc", "away_team", "home_team", "book_spread", "my_spread", "actual_margin", "ats_result"]].head() if not games.empty else games


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,ats_result
0,2026-04-20 00:00:00+00:00,MIN,DEN,1.0,-9.885484,-5,loss
1,2026-04-20 00:00:00+00:00,ATL,NY,NaN,-3.225038,-1,loss
2,2026-04-20 00:00:00+00:00,TOR,CLE,3.5,-6.797049,10,win
3,2026-04-21 00:00:00+00:00,HOU,LAL,-9.5,1.590512,7,win
4,2026-04-21 00:00:00+00:00,PHI,BOS,7.5,-6.948005,-14,loss


In [5]:
def summarize_ats(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"league": LEAGUE, "season": season, "graded_games": 0, "note": "No scored games with predictions available."}
    wins = int((df["ats_result"] == "win").sum())
    losses = int((df["ats_result"] == "loss").sum())
    pushes = int((df["ats_result"] == "push").sum())
    risked = wins + losses
    net_units = wins * WIN_PROFIT_AT_MINUS_110 - losses
    return {
        "league": LEAGUE,
        "season": season,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "graded_games": wins + losses + pushes,
        "ats_pct": wins / risked if risked else None,
        "flat_roi_at_minus_110": net_units / risked if risked else None,
        "net_units": net_units,
    }

summary = summarize_ats(games)
if not games.empty:
    summary["graded_missing_book_spread"] = int(games["book_spread"].isna().sum())
    book_present = games[games["book_spread"].notna()]
    summary["ats_where_book_line_present"] = summarize_ats(book_present) if not book_present.empty else None

summary


{'league': 'NBA',
 'season': 2025,
 'wins': 15,
 'losses': 16,
 'pushes': 0,
 'graded_games': 31,
 'ats_pct': 0.4838709677419355,
 'flat_roi_at_minus_110': -0.07624633431085043,
 'net_units': -2.3636363636363633,
 'graded_missing_book_spread': 8,
 'ats_where_book_line_present': {'league': 'NBA',
  'season': 2025,
  'wins': 12,
  'losses': 11,
  'pushes': 0,
  'graded_games': 23,
  'ats_pct': 0.5217391304347826,
  'flat_roi_at_minus_110': -0.00395256916996051,
  'net_units': -0.09090909090909172}}

In [6]:
if not games.empty:
    games_for_edge = games[games["book_spread"].notna()].copy()
    games_for_edge["edge_bucket"] = pd.cut(games_for_edge["edge_vs_book"], bins=EDGE_BUCKETS)
    edge_summary = (
        games_for_edge.groupby("edge_bucket", observed=True)
        .agg(
            games=("game_id", "count"),
            wins=("ats_result", lambda s: int((s == "win").sum())),
            losses=("ats_result", lambda s: int((s == "loss").sum())),
            pushes=("ats_result", lambda s: int((s == "push").sum())),
            avg_edge=("edge_vs_book", "mean"),
            avg_cover_margin=("cover_margin", "mean"),
        )
        .reset_index()
    )
    edge_summary["risked_games"] = edge_summary["wins"] + edge_summary["losses"]
    edge_summary["ats_pct"] = edge_summary["wins"] / edge_summary["risked_games"].replace(0, np.nan)
    edge_summary["roi"] = (edge_summary["wins"] * WIN_PROFIT_AT_MINUS_110 - edge_summary["losses"]) / edge_summary["risked_games"].replace(0, np.nan)
else:
    edge_summary = pd.DataFrame()

edge_summary


,edge_bucket,games,wins,losses,pushes,avg_edge,avg_cover_margin,risked_games,ats_pct,roi
0,"(-99, -8]",8,3,5,0,-12.250013,-7.187513,8,0.375000,-0.284091
1,"(-8, -5]",5,4,1,0,-5.749476,8.550524,5,0.800000,0.527273
2,"(-5, -3]",1,0,1,0,-4.099626,-2.599626,1,0.000000,-1.000000
3,"(-1, 1]",2,2,0,0,-0.458802,13.041198,2,1.000000,0.909091
4,"(1, 3]",1,0,1,0,1.451965,-7.048035,1,0.000000,-1.000000
5,"(3, 5]",2,0,2,0,3.530077,-3.969923,2,0.000000,-1.000000
6,"(5, 8]",1,1,0,0,6.404742,11.904742,1,1.000000,0.909091
7,"(8, 99]",3,2,1,0,12.168113,-4.331887,3,0.666667,0.272727


In [7]:
if not games.empty:
    biggest_hits = games.sort_values("cover_margin", ascending=False).head(10)
    biggest_misses = games.sort_values("cover_margin", ascending=True).head(10)
    display_cols = ["game_time_utc", "away_team", "home_team", "book_spread", "my_spread", "actual_margin", "cover_margin", "model_version"]
    print("Biggest hits")
    display(biggest_hits[display_cols])
    print("Biggest misses")
    display(biggest_misses[display_cols])


Biggest hits


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,cover_margin,model_version
26,2026-04-28 00:00:00+00:00,ATL,NY,NaN,-4.865358,29,24.134642,v3
8,2026-04-23 00:00:00+00:00,CLE,TOR,3.5,-2.868005,22,19.131995,v3
21,2026-04-26 00:00:00+00:00,LAL,HOU,-2.5,-2.483638,19,16.516362,v3
25,2026-04-28 00:00:00+00:00,POR,SA,NaN,-5.291050,19,13.708950,v3
10,2026-04-23 00:00:00+00:00,DEN,MIN,1.5,-3.773887,17,13.226113,v3
14,2026-04-25 00:00:00+00:00,DEN,MIN,-10.5,-4.095258,16,11.904742,v3
7,2026-04-22 00:00:00+00:00,ORL,DET,2.5,-3.232948,15,11.767052,v3
16,2026-04-25 00:00:00+00:00,DET,ORL,2.5,1.566034,8,9.566034,v3
3,2026-04-21 00:00:00+00:00,HOU,LAL,-9.5,1.590512,7,8.590512,v3
6,2026-04-22 00:00:00+00:00,PHX,OKC,9.5,-5.740951,13,7.259049,v3


Biggest misses


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,cover_margin,model_version
19,2026-04-26 00:00:00+00:00,BOS,PHI,-11.5,3.916427,-32,-28.083573,v3
27,2026-04-28 00:00:00+00:00,PHI,BOS,5.5,-6.871993,-16,-22.871993,v3
4,2026-04-21 00:00:00+00:00,PHI,BOS,7.5,-6.948005,-14,-20.948005,v3
18,2026-04-26 00:00:00+00:00,SA,POR,NaN,2.254627,-21,-18.745373,v3
15,2026-04-25 00:00:00+00:00,NY,ATL,NaN,0.676460,-16,-15.323540,v3
0,2026-04-20 00:00:00+00:00,MIN,DEN,1.0,-9.885484,-5,-14.885484,v3
11,2026-04-24 00:00:00+00:00,SA,POR,NaN,-0.111991,-12,-12.111991,v3
17,2026-04-25 00:00:00+00:00,OKC,PHX,10.5,2.206844,-12,-9.793156,v3
13,2026-04-24 00:00:00+00:00,LAL,HOU,-4.5,-3.048035,-4,-7.048035,v3
5,2026-04-21 00:00:00+00:00,POR,SA,NaN,-3.824298,-3,-6.824298,v3


In [8]:
if not games.empty and games["my_home_win_prob"].notna().sum() >= 20:
    games["win_prob_bucket"] = pd.cut(games["my_home_win_prob"], bins=np.linspace(0, 1, 11), include_lowest=True)
    calibration = (
        games.groupby("win_prob_bucket", observed=True)
        .agg(games=("game_id", "count"), avg_pred=("my_home_win_prob", "mean"), actual_home_win_rate=("home_win_actual", "mean"))
        .reset_index()
    )
else:
    calibration = pd.DataFrame({"note": ["Not enough completed games with win probabilities for calibration buckets."]})

calibration


,win_prob_bucket,games,avg_pred,actual_home_win_rate
0,"(0.3, 0.4]",1,0.337681,0.000000
1,"(0.4, 0.5]",9,0.461955,0.333333
2,"(0.5, 0.6]",8,0.562961,0.625000
3,"(0.6, 0.7]",10,0.649349,0.700000
4,"(0.7, 0.8]",3,0.727703,0.666667
